# 04 Stop + Re-entry vs Hold

Directly compares stop -> sell -> pay tax -> re-enter -> recover against holding through the bear drawdown and recovery.

**Plain English:**
This notebook tests whether selling on a stop, paying tax, buying back lower, and riding the rebound beats doing nothing.

**This answers the question:** "Would I be better off exiting and re-entering, or just holding through the drop?"

Example:
If a 5% stop sells at 332.50, then a -30% bear low is 245 and re-entry slippage is 5%, the model buys back at 257.25 and compares that recovery result with holding.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from tax_risk_sim import (
    build_after_tax_sensitivity,
    build_bear_recovery_cases,
    build_bear_recovery_table,
    build_probability_weighted_scenarios,
    build_stop_benchmark,
    compare_stop_reentry_vs_hold,
    sell_today_baseline,
)

pd.options.display.float_format = "{:,.2f}".format

## Inputs

These inputs define the position, stop levels, bear scenarios, and re-entry assumptions.

**Plain English:**
This is where you set the rules for selling, buying back, and how imperfect the buy-back is.

**This answers the question:** "What stop-and-re-entry strategy am I testing?"

Example:
`reentry_slippage_from_bear_low = 0.05` means if the bear low is 245, re-entry happens at 257.25.

### Input Explanations

`reentry_slippage_from_bear_low` models missing the exact low. A value of `0.05` means re-entry happens 5% above the bear low.

**Plain English:**
This makes the buy-back less perfect.

**This answers the question:** "What if I do not buy back at the absolute bottom?"

Example:
If the bear low is 245 and slippage is 5%, re-entry price is 257.25.

`transaction_cost_rate` models execution cost on both the stop sale and the re-entry purchase.

**Plain English:**
This removes a small cost for trading.

**This answers the question:** "What if selling and buying back are not free?"

Example:
A 0.5% transaction cost reduces cash on the sale and again on the buy-back.

`allow_fractional_reentry_shares` controls whether re-entry can buy fractional shares. `False` means whole shares only and leftover cash remains uninvested.

**Plain English:**
If this is false, the model rounds down the number of shares you can buy.

**This answers the question:** "What if I can only buy whole shares?"

Example:
If cash can buy 37.86 shares, the model buys 37 shares and keeps the leftover cash.

The bear drawdown inputs define the scenario grid used by the heatmaps.

**Plain English:**
They create the market-drop columns shown in the heatmaps.

**This answers the question:** "Which bear markets should the strategy be tested against?"

Example:
-5% to -60% creates columns from Bear -5% through Bear -60%.

In [ ]:
from inputs import (
    shares,
    current_price,
    cost_basis_per_share,
    capital_gains_tax_rate,
    stop_loss_drops,
    reentry_slippage_from_bear_low,
    transaction_cost_rate,
    allow_fractional_reentry_shares,
    bear_drawdown_start,
    bear_drawdown_end,
    bear_drawdown_step,
    bear_recovery_multiplier,
    bear_min_recovery_return,
    bear_max_recovery_return,
    bear_base_recovery_probability,
    bear_min_recovery_probability,
)

bear_recovery_cases = build_bear_recovery_cases(
    start=bear_drawdown_start,
    end=bear_drawdown_end,
    step=bear_drawdown_step,
    recovery_multiplier=bear_recovery_multiplier,
    min_recovery_return=bear_min_recovery_return,
    max_recovery_return=bear_max_recovery_return,
    base_recovery_probability=bear_base_recovery_probability,
    min_recovery_probability=bear_min_recovery_probability,
)

bear_recovery_cases

In [ ]:
pd.Series(
    {
        "Shares held": shares,
        "Current price": f"${current_price:,.2f}",
        "Cost basis per share": f"${cost_basis_per_share:,.2f}",
        "Capital gains tax rate": f"{capital_gains_tax_rate:.0%}",
        "Stop-loss levels tested": ", ".join(f"{d:.0%}" for d in stop_loss_drops),
        "Bear drawdown range": (
            f"{bear_drawdown_start:.0%} to {bear_drawdown_end:.0%}"
            f" in {bear_drawdown_step:.0%} steps"
        ),
        "Recovery multiplier": bear_recovery_multiplier,
        "Recovery return range": (
            f"{bear_min_recovery_return:.0%} to {bear_max_recovery_return:.0%}"
        ),
        "Recovery probability range": (
            f"{bear_min_recovery_probability:.0%} to {bear_base_recovery_probability:.0%}"
        ),
        "Re-entry slippage from bear low": f"{reentry_slippage_from_bear_low:.0%}",
        "Transaction cost per trade": f"{transaction_cost_rate:.1%}",
        "Fractional shares allowed": allow_fractional_reentry_shares,
    },
    name="Inputs",
)

## Build Baseline Tables

Builds the sell-today baseline, stop benchmark table, bear recovery table, and numeric bear-case ordering.

**Plain English:**
This prepares the data that the heatmaps need.

**This answers the question:** "What are the stop levels and bear scenarios before comparing strategies?"

Example:
It calculates that a 20% stop is 280 and that a -30% bear low is 245.

In [ ]:
baseline = sell_today_baseline(
    shares,
    current_price,
    cost_basis_per_share,
    capital_gains_tax_rate,
)
stop_benchmark_df = build_stop_benchmark(
    stop_loss_drops,
    shares,
    current_price,
    cost_basis_per_share,
    capital_gains_tax_rate,
)
bear_recovery_df = build_bear_recovery_table(
    bear_recovery_cases,
    shares,
    current_price,
    cost_basis_per_share,
    capital_gains_tax_rate,
)
bear_case_order = bear_recovery_df.sort_values("drawdown", ascending=False)["case"].tolist()

baseline

## Combined Benchmark Heatmap

Shows whether each bear scenario's assumed recovery clears the required recovery from each stop level.

**Plain English:**
This is a quick yes/no map showing whether the assumed rebound is big enough for each stop and bear case.

**This answers the question:** "Is the assumed recovery large enough to justify tolerating this stop level?"

Example:
If a 20% stop needs a 25% rebound and the bear scenario assumes a 45% rebound, the heatmap marks it as recovery enough.

In [ ]:
combined_rows = []
for _, stop in stop_benchmark_df.iterrows():
    for _, scenario in bear_recovery_df.iterrows():
        combined_rows.append({
            "stop_loss_drop": stop["stop_loss_drop"],
            "stop_price": stop["stop_price"],
            "bear_case": scenario["case"],
            "bear_drawdown": scenario["drawdown"],
            "covered_by_stop": abs(scenario["drawdown"]) >= stop["stop_loss_drop"],
            "required_recovery_from_stop_to_match_selling_today": stop["required_recovery_from_stop_to_match_selling_today"],
            "assumed_recovery_from_bear_low": scenario["recovery_return_from_low"],
            "assumed_recovery_clears_required_recovery": scenario["recovery_return_from_low"] >= stop["required_recovery_from_stop_to_match_selling_today"],
            "recovery_probability": scenario["recovery_probability"],
            "expected_difference_vs_selling_today_if_hold_for_recovery": scenario["expected_difference_vs_selling_today"],
        })

stop_vs_bear_df = pd.DataFrame(combined_rows)
stop_vs_bear_df

### Combined Benchmark Column Explanations

`covered_by_stop` means the bear drawdown is deep enough that the stop would trigger.

**Plain English:**
This says whether the stop gets hit before or during that bear scenario.

**This answers the question:** "Would this stop actually sell in this bear market?"

Example:
A 20% stop is covered by a -30% bear scenario, but not by a -10% bear scenario.

`required_recovery_from_stop_to_match_selling_today` is the rebound needed from the stop price to equal selling today after tax.

**Plain English:**
This is the comeback needed from the stop price.

**This answers the question:** "How much must the price bounce after this stop level?"

Example:
A 20% stop at 280 needs a 25% rebound to get back to 350.

`assumed_recovery_from_bear_low` is the scenario's assumed rebound from the bear low.

**Plain English:**
This is the bounce the scenario assumes happens after the low.

**This answers the question:** "How much recovery does this bear scenario assume?"

Example:
A -30% bear scenario may assume a 45% recovery from the low.

`assumed_recovery_clears_required_recovery` is `True` when the assumed rebound is large enough to clear the stop benchmark.

**Plain English:**
`True` means the assumed bounce is enough; `False` means it is not.

**This answers the question:** "Is the assumed recovery big enough for this stop level?"

Example:
45% assumed recovery clears a 25% required recovery.

In [ ]:
heatmap_df = stop_vs_bear_df.pivot_table(
    index="stop_loss_drop",
    columns="bear_case",
    values="assumed_recovery_clears_required_recovery",
    aggfunc="first",
).reindex(columns=bear_case_order).astype(int)

fig, ax = plt.subplots(figsize=(16, 6))
im = ax.imshow(heatmap_df.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_title("Combined Benchmark Heatmap: Does Assumed Bear Recovery Clear Required Recovery?")
ax.set_xlabel("Bear scenario")
ax.set_ylabel("Candidate stop-loss trigger: drop from today's price")
ax.set_xticks(np.arange(len(heatmap_df.columns)))
ax.set_xticklabels(heatmap_df.columns, rotation=90, ha="center", fontsize=7)
ax.set_yticks(np.arange(len(heatmap_df.index)))
ax.set_yticklabels([f"{value:.0%}" for value in heatmap_df.index])
cbar = fig.colorbar(im, ax=ax, ticks=[0, 1])
cbar.ax.set_yticklabels(["Recovery too low", "Recovery enough"])
plt.tight_layout()
plt.show()

## Stop + Re-entry Strategy Test

The key column is `stop_reentry_advantage_vs_hold_after_recovery`. Positive means stop + re-entry beats holding through that bear recovery.

**Plain English:**
This is the main comparison: sell on stop and buy back lower versus just keep holding.

**This answers the question:** "After tax, slippage, whole-share rounding, and recovery, which strategy leaves me with more money?"

Example:
For Bear -30%, a 5% stop sells at 332.50, re-enters at 257.25, buys 37 whole shares, and compares the recovery value with holding the original 35 shares.

In [ ]:
stop_reentry_vs_hold_df = compare_stop_reentry_vs_hold(
    stop_benchmark_df,
    bear_recovery_df,
    shares,
    cost_basis_per_share,
    capital_gains_tax_rate,
    reentry_slippage_from_bear_low=reentry_slippage_from_bear_low,
    transaction_cost_rate=transaction_cost_rate,
    allow_fractional_reentry_shares=allow_fractional_reentry_shares,
)

stop_reentry_vs_hold_df

### Stop + Re-entry Column Explanations

`reentry_price` is the bear low adjusted by re-entry slippage.

**Plain English:**
This is the price where the model buys back.

**This answers the question:** "At what price do I re-enter?"

Example:
A 245 bear low with 5% slippage gives a 257.25 re-entry price.

`raw_reentry_shares_before_rounding` is how many shares the after-tax cash could buy before whole-share rounding.

**Plain English:**
This is the exact share count before rounding down.

**This answers the question:** "How many shares could my cash theoretically buy?"

Example:
If cash could buy 37.86 shares, this column shows 37.86.

`reentry_shares_after_tax_and_costs` is the actual modeled share count after tax, costs, and whole-share rounding.

**Plain English:**
This is how many whole shares the model actually buys.

**This answers the question:** "How many shares do I own after re-entry?"

Example:
37.86 raw shares becomes 37 whole shares.

`leftover_cash_after_reentry` is uninvested cash left after buying whole shares.

**Plain English:**
This is the cash that could not buy another whole share.

**This answers the question:** "How much cash is left sitting aside?"

Example:
If 37 shares are bought and 221.90 remains, this column shows 221.90.

`stop_reentry_advantage_vs_hold_after_recovery` is the key result: positive means stop + re-entry beats holding through that bear recovery.

**Plain English:**
This is the extra money from the stop-and-buy-back strategy compared with holding.

**This answers the question:** "Did stop + re-entry make me richer than holding?"

Example:
If this column is 3,036.77, stop + re-entry beat holding by 3,036.77 in that scenario.

In [ ]:
best_stop_reentry_by_case = (
    stop_reentry_vs_hold_df[stop_reentry_vs_hold_df["stop_triggers"]]
    .sort_values("stop_reentry_advantage_vs_hold_after_recovery", ascending=False)
    .groupby("bear_case", as_index=False)
    .first()
    [[
        "bear_case",
        "bear_drawdown",
        "stop_loss_drop",
        "stop_price",
        "reentry_price",
        "recovery_price",
        "raw_reentry_shares_before_rounding",
        "reentry_shares_after_tax_and_costs",
        "leftover_cash_after_reentry",
        "hold_after_recovery_after_tax",
        "stop_reentry_after_recovery_value",
        "stop_reentry_advantage_vs_hold_after_recovery",
        "stop_reentry_advantage_vs_hold_after_recovery_pct",
    ]]
)

best_stop_reentry_by_case

In [ ]:
strategy_heatmap_df = stop_reentry_vs_hold_df.pivot_table(
    index="stop_loss_drop",
    columns="bear_case",
    values="stop_reentry_advantage_vs_hold_after_recovery_pct",
    aggfunc="first",
).reindex(columns=bear_case_order)

fig, ax = plt.subplots(figsize=(16, 6))
max_abs = np.nanmax(np.abs(strategy_heatmap_df.values))
max_abs = max(max_abs, 0.01)
im = ax.imshow(strategy_heatmap_df.values, cmap="RdYlGn", vmin=-max_abs, vmax=max_abs, aspect="auto")
ax.set_title("Stop + Re-entry Advantage vs Holding Through Bear Recovery")
ax.set_xlabel("Bear scenario")
ax.set_ylabel("Candidate stop-loss trigger: drop from today's price")
ax.set_xticks(np.arange(len(strategy_heatmap_df.columns)))
ax.set_xticklabels(strategy_heatmap_df.columns, rotation=90, ha="center", fontsize=7)
ax.set_yticks(np.arange(len(strategy_heatmap_df.index)))
ax.set_yticklabels([f"{value:.0%}" for value in strategy_heatmap_df.index])
cbar = fig.colorbar(im, ax=ax)
cbar.ax.set_ylabel("Advantage vs hold after recovery")
subtitle = (
    f"Re-entry slippage from bear low: {reentry_slippage_from_bear_low:.1%} | "
    f"Transaction cost per trade: {transaction_cost_rate:.1%} | "
    f"Fractional shares: {allow_fractional_reentry_shares}"
)
ax.text(0.5, 1.12, subtitle, transform=ax.transAxes, ha="center", fontsize=9)
plt.tight_layout()
plt.show()